# 83장 · 멱등적인 완료 상태 갱신

같은 완료 요청을 다시 보냈더니 체크가 풀렸습니다. 재전송 뒤에도 결과가 같아야 하는 조건을 멱등성이라고 합니다.

완료 요청을 두 번 보냈을 때 체크 표시가 풀리는 오류를 재현합니다. 현재 상태를 뒤집는 구현과 요청의 completed 값을 대입하는 구현을 같은 입력으로 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 토글 방식과 중복 요청 재현

제공된 faulty 저장소는 completed=True 요청을 ‘현재 값을 뒤집기’로 처리합니다. 같은 ID, 같은 초기 상태, 같은 입력을 사용해야 수정 전후 결과를 비교할 수 있습니다.

**같은 완료 요청의 반복과 멱등성**　완료 요청을 두 번 보냈을 때 토글 구현의 최종값은 False, 목표값을 대입한 구현은 True입니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter83/index.html#practice-space)


### 동일한 요청을 두 번 보내기

<p>결함을 보려면 같은 딕셔너리에 목표 True를 두 번 전달해야 합니다. 호출할 때마다 새 딕셔너리를 만들면 첫 번째 요청만 반복하는 셈이어서 두 번째 호출의 문제를 발견할 수 없습니다.</p>


### 셀 01 · 반복 요청으로 토글 오류 재현

아래 결함은 수업용으로 의도해 넣었습니다. 두 번째 결과를 첫 번째와 비교하세요.


In [1]:
def faulty_update(item, completed):
    item["completed"] = not item["completed"]
    return item["completed"]

item = {"id": 1, "title": "복습", "completed": False}
print(faulty_update(item, True))
print(faulty_update(item, True))

True
False


같은 행에 True를 두 번 전달했는데 두 번째 호출 뒤에 False가 남습니다. 첫 번째 응답을 받지 못해 같은 요청을 다시 보낸 상황에서도 이 문제가 생길 수 있습니다.

<div class="example-table"><table><caption>동일한 완료 요청을 두 번 보낸 결과</caption><thead><tr><th scope="col">수정 방식</th><th scope="col">초기 상태</th><th scope="col">첫 요청</th><th scope="col">재전송</th></tr></thead><tbody><tr><td>현재 값 반전</td><td>False</td><td>True</td><td>False</td></tr><tr><td>목표값 대입</td><td>False</td><td>True</td><td>True</td></tr></tbody></table></div>


#### 같은 완료 요청이 두 번째에 미완료를 만드는 경로

```mermaid
flowchart TD
ninput["완료 요청"]
  nflip["상태 반전"]
  nyes["완료"]
  nno["미완료"]
  ninput --> nflip
  nflip --> nyes
  nyes --> ninput
  nflip --> nno
```

faulty_update는 전달받은 목표 값 대신 현재 값을 뒤집습니다. 요청이 두 번 들어왔을 때의 실제 출력을 읽어 보세요.

### 호출 횟수에 따라 바뀌는 토글

<p>faulty_update는 전달된 completed 인자를 사용하지 않습니다. 현재 값에 not을 적용하므로 이미 완료된 행도 같은 요청을 한 번 더 받으면 미완료가 됩니다.</p>


## 목표 상태 대입과 회귀 검사

수정 범위를 완료 필드 반영 한 줄에 둡니다. PATCH 요청이 들어온 횟수에 관계없이 명시적으로 전달받은 상태를 저장해야 합니다.


### 과제 02 · 목표 상태 대입의 멱등성

completed에 요청값을 저장하는 함수를 작성하세요. 같은 행에 True를 두 번 전달하고 두 번째 호출 뒤에 반환값과 실제 저장값을 함께 출력하세요.


In [2]:
def update(item, completed):
    # 호출자가 넘긴 딕셔너리에 목표 상태를 반영하고 저장값을 반환하세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def update(item, completed):
    item[&quot;completed&quot;] = completed
    return item[&quot;completed&quot;]

item = {&quot;id&quot;: 1, &quot;title&quot;: &quot;복습&quot;, &quot;completed&quot;: False}
update(item, True)
print(update(item, True), item[&quot;completed&quot;])</code></pre>
<p>실행 결과</p><pre><code>True True</code></pre>
<p>첫 번째 True는 두 번째 호출의 반환값이고, 그 옆의 True는 딕셔너리에서 다시 읽은 값입니다. 반환만 하고 행을 수정하지 않은 함수라면 두 값이 다릅니다.</p>
</details>


#### 같은 완료 요청이 두 번 도착하면

<mark>토글은 두 번 실행하면 원래대로 돌아가지만 목표값 대입은 같은 상태를 유지합니다.</mark> 처음부터 두 번 연속 요청을 보내고 두 번째 응답까지 보세요. 한 번만 시험하면 두 구현 모두 완료로 보여 차이를 놓칩니다.

선수 개념 · 81장 FastAPI로 구현하는 할 일 CRUD

### 목표 상태의 직접 대입

<p>수정한 함수는 호출자가 원하는 값을 item["completed"]에 대입해야 합니다. 같은 True를 여러 번 대입해도 저장값은 True로 유지되고, False를 전달한 경우에만 완료가 해제됩니다.</p>


### 셀 03 · 완료 해제의 회귀 테스트

과제 02에서 완성한 update에 True, True, False를 보내고 매번 반환값과 같은 행의 저장값을 확인합니다.

과제 02에서 작성한 update를 실행한 뒤 이 셀로 오세요. 같은 True 요청을 두 번 보내고 이어 False로 되돌립니다. 셀을 고쳤다면 함수 정의 셀부터 다시 실행하세요.


In [3]:
row = {"id": 5, "title": "복습", "completed": False}
first = update(row, True)
after_first = row.copy()
second = update(row, True)
after_second = row.copy()
last = update(row, False)
if first is None and second is None and last is None:
    print("과제 02의 update를 완성하고 정의 셀부터 다시 실행하세요.")
else:
    assert first is True and second is True, "같은 완료 요청을 반복해도 True를 반환해야 합니다."
    expected = {"id": 5, "title": "복습", "completed": True}
    assert after_first == expected and after_second == expected, "두 완료 요청 모두 원래 행에 저장되어야 합니다."
    assert last is False, "완료 해제 요청은 False를 반환해야 합니다."
    assert row == {"id": 5, "title": "복습", "completed": False}, "같은 행의 ID·제목·완료 값을 확인하세요."
    print("검사 통과", row)

과제 02의 update를 완성하고 정의 셀부터 다시 실행하세요.


시작본의 pass는 None을 반환합니다. 세 번 모두 None이면 검사를 통과 처리하지 않고 과제 02를 완성하라는 안내를 출력합니다. 정답을 넣고 과제 02와 이 셀을 다시 실행했을 때의 마지막 줄은 다음과 같습니다.

검사 통과 {'id': 5, 'title': '복습', 'completed': False}

완료 요청을 두 번 보낸 뒤 해제했으므로 마지막 값은 False입니다. 요청할 때마다 상태를 뒤집는 구현은 두 번째 True 검사에서 멈춥니다.


### 반환값과 실제 저장값을 따로 검사하기

<p>first와 second의 반환값이 모두 True인지 보고, 두 시점에 복사한 행의 id·title·completed도 대조합니다. True를 반환하기만 하고 딕셔너리를 고치지 않은 함수라면 행 비교에서 드러납니다.</p>


## 삭제 뒤 순번이 바뀐 목록에서 잘못된 수정 재현하기

앞에서는 같은 완료 요청을 반복했을 때 값이 뒤집히는 오류를 고쳤습니다. 이번 오류는 응답이 성공인데도 다른 항목이 완료됩니다. 재현 입력을 먼저 만들고 Codex가 제안한 수정이 이웃 항목까지 보존하는지 검사합니다.

### 셀 05 · ID를 리스트 위치로 사용한 수정 코드 실행하기

삭제가 있었던 목록에는 ID 2와 7만 남았습니다. ID 2를 완료하려고 다음 함수를 호출한 뒤 두 행의 상태를 출력하세요.

In [4]:
def after_deletion():
    return [{"id": 2, "completed": False}, {"id": 7, "completed": False}]

def update_at_position(rows, todo_id, value):
    rows[todo_id - 1]["completed"] = value

repro = after_deletion()
update_at_position(repro, 2, True)
print([(entry["id"], entry["completed"]) for entry in repro])

[(2, False), (7, True)]


출력은 2번 False, 7번 True입니다. 2에서 1을 빼면 리스트 위치 1이 되고, 삭제가 있었던 이 목록에서 그 위치는 ID 7입니다. 예외가 발생하지 않는 잘못된 수정입니다.

### 과제 06 · ID가 일치하는 행만 바꾸는 회귀 검사 작성하기

수정 대상이 있으면 True, 없으면 False를 반환하게 고치세요. ID 2에 두 차례 같은 완료 요청을 보낸 뒤 없는 9번도 시도합니다. 마지막에는 7번이 여전히 미완료인지 assert로 검사하세요.

In [5]:
# change_by_id(rows, todo_id, value)와 회귀 검사를 작성하세요.
# after_deletion()으로 매번 새 재현 목록을 만드세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def change_by_id(rows, todo_id, value):
    for row in rows:
        if row[&quot;id&quot;] == todo_id:
            row[&quot;completed&quot;] = value
            return True
    return False

checked_rows = after_deletion()
assert change_by_id(checked_rows, 2, True)
assert change_by_id(checked_rows, 2, True)
assert not change_by_id(checked_rows, 9, True)
assert checked_rows == [{&quot;id&quot;: 2, &quot;completed&quot;: True}, {&quot;id&quot;: 7, &quot;completed&quot;: False}]
print(checked_rows)</code></pre>
<p>실행 결과</p><pre><code>[{&#x27;id&#x27;: 2, &#x27;completed&#x27;: True}, {&#x27;id&#x27;: 7, &#x27;completed&#x27;: False}]</code></pre>
<p>이번에는 2번만 True입니다. 두 번째 요청과 없는 번호에 대한 검사도 통과한 뒤 목록이 출력됐습니다. 수정 전후의 재현 입력을 그대로 유지해야 다른 조건을 우연히 통과한 경우를 가릴 수 있습니다.</p>
</details>

<mark>수정 성공 응답과 실제로 바뀐 행의 ID를 대조하세요.</mark> 연속 ID 1, 2만 넣은 테스트는 ID와 위치가 어긋나는 결함을 드러내지 못합니다. 삭제 이후의 목록을 Codex에 재현 입력으로 전달하세요.

다음 요구 변경에서도 기존 행과 건드리지 않아야 할 이웃 행을 검증 목록에 남깁니다.

## 돌아보기

수정한 함수에 False를 보내면 이미 완료된 행의 체크가 풀려야 합니다. 항상 True를 반환하는 구현은 완료 요청 두 번만으로는 걸러지지 않으므로 다음 셀에서 해제 후 값도 확인합니다.
